# 04 — LLM Reporting (Bee-A-Hero, final stage)

Turns the CV + ML numbers into a plain-language grower report.
Pipeline: **Data -> CV -> ML -> LLM**. This notebook runs the last two links:

1. **ML apply**: fitted fruit-set curve (`models/dose_response_v11.json`) applied to the CV
   tracker export (`test_video_result/csv/ALL_landings.csv`) -> a yield band (`models/yield_report.json`).
2. **LLM report**: `src/llm_reporting/generate.py` reads the grounded CV + ML numbers and writes a
   farmer-friendly Markdown report.

The report generator picks a back-end automatically: **Claude** (`claude-opus-4-8`) when the
`anthropic` SDK + a credential are present (a grounded narrative citing only measured facts), else an
**offline template** from the same facts (no network call), so the pipeline runs from a clean checkout.

Setup (once): `pip install -r src/llm_reporting/requirements-llm.txt`


## 0. Locate the repo root (so `import src` works from anywhere)


In [1]:
import os, sys
from pathlib import Path
here = Path.cwd()
for cand in [here, *here.parents]:
    if (cand / 'src' / 'config.py').exists():
        os.chdir(cand); sys.path.insert(0, str(cand)); break
REPO = Path.cwd()
print('repo root:', REPO)


repo root: /home/manheim666/Desktop/Bee-A-Hero


## 1. ML apply — tracker export -> yield band


In [2]:
import subprocess, sys
# apply-only: loads the committed curve and applies it to the CV tracker CSV (no training data needed)
print(subprocess.run([sys.executable, '-m', 'src.ml_models.train'],
                     capture_output=True, text=True).stdout)


[info] training dataset not found — running apply-only from dose_response_v11.json
[apply-only] Loaded 'pomegranate' curve from dose_response_v11.json (F0=0.447 Fmax=0.687 k=0.0067)
[apply-only] Applying to tracker output ALL_landings.csv ...
      gates applied on real data: ['dwell']
      fruit set at mean dose = 0.46 [0.45, 0.46]
      yield = 164 kg/tree [155, 174]
      wrote yield report -> yield_report.json



## 2. Inspect the grounded facts the report is allowed to use


In [3]:
import json
from src.llm_reporting.generate import collect_facts
facts = collect_facts()
print(json.dumps(facts, indent=2))


{
  "crop": "pomegranate",
  "sources": [
    "test_video_result/csv/ALL_flower_summary.csv",
    "test_video_result/csv/ALL_landings.csv",
    "models/dose_response_v11.json",
    "models/yield_report.json"
  ],
  "n_flowers": 60,
  "pollination_score_total": 1756.3,
  "n_videos": 20,
  "n_episodes": 155,
  "n_real_landings": 31,
  "real_by_type": {
    "honeybee": 21,
    "bee": 4,
    "butterfly": 3,
    "fly": 2,
    "bug": 1
  },
  "n_honeybee_real": 21,
  "gates": {
    "dwell_min_s": 2.0,
    "vel_max": 0.05,
    "frac_min": 0.6
  },
  "curve": {
    "F0": 0.4469,
    "Fmax": 0.6866,
    "k": 0.00672
  },
  "mean_effective_dose": 5.6779,
  "fruit_set_mean": 0.4558736000451767,
  "fruit_set_ci95": [
    0.44985231228745576,
    0.46240528976180345
  ],
  "yield_kg_mean": 163.8675,
  "yield_kg_ci95": [
    154.7925,
    174.0
  ],
  "yield_note": "apply-only: bootstrap draws resampled from stored CIs (training dataset not present); yield band is illustrative"
}


## 3. Generate the report

Force the offline template here for reproducibility without a key. Drop `force_offline=True`
(and install/authenticate `anthropic`) to get the Claude-written narrative instead.


In [4]:
from src.llm_reporting.generate import generate_report
report, backend = generate_report(Path('docs/results/cv/pollination_report.md'),
                                  force_offline=True)
print(f'backend: {backend}\n')
print(report)


[llm] wrote offline report -> docs/results/cv/pollination_report.md
backend: offline

# Pollination report

**Crop:** pomegranate  ·  **Footage analysed:** 20 clips

## What the camera saw
- **31 real pollination visits** (insects that stayed >= 2 s) across **60 flowers**, out of 155 total landing episodes.
- **21 of those were honeybees** — the strongest pollinators, weighted most heavily in the score.
- Visits by insect type: honeybee 21 . bee 4 . butterfly 3 . fly 2 . bug 1.
- Combined **pollination score: 1756.3** (honeybee visits count 10x).

## What it means for fruit set
- Estimated **fruit set: 46%** (95% range 45%-46%) at the observed visit rate.
- Illustrative **yield: 164 kg/tree** (95% range 155-174).
- *apply-only: bootstrap draws resampled from stored CIs (training dataset not present); yield band is illustrative*

## Caveats
- Visits are measured directly; fruit set and yield are model estimates that need field ground-truth before they are firm.
- The honeybee split is p

## 4. Render it

The same Markdown, formatted — what a grower would receive.


In [5]:
from IPython.display import Markdown
Markdown(report)


# Pollination report

**Crop:** pomegranate  ·  **Footage analysed:** 20 clips

## What the camera saw
- **31 real pollination visits** (insects that stayed >= 2 s) across **60 flowers**, out of 155 total landing episodes.
- **21 of those were honeybees** — the strongest pollinators, weighted most heavily in the score.
- Visits by insect type: honeybee 21 . bee 4 . butterfly 3 . fly 2 . bug 1.
- Combined **pollination score: 1756.3** (honeybee visits count 10x).

## What it means for fruit set
- Estimated **fruit set: 46%** (95% range 45%-46%) at the observed visit rate.
- Illustrative **yield: 164 kg/tree** (95% range 155-174).
- *apply-only: bootstrap draws resampled from stored CIs (training dataset not present); yield band is illustrative*

## Caveats
- Visits are measured directly; fruit set and yield are model estimates that need field ground-truth before they are firm.
- The honeybee split is provisional and currently over-calls honeybee.

---
*Sources: test_video_result/csv/ALL_flower_summary.csv, test_video_result/csv/ALL_landings.csv, models/dose_response_v11.json, models/yield_report.json.*